# SinhalaCheck — Module 1: NLP Content Credibility Analysis
### Corrected-Data Retraining, Comparative Model Analysis & Statistical Validation

**Project:** R26-IT-158 | **Module 1** | Kaweeshwara P.D.S. (IT22331304) | SLIIT

---

### What changed from the first run, and why

The first run established two things: the CSV distribution of the LIRNEasia corpus has lost all
Sinhala text, and training on the intact XLSX distribution instead raises macro-F1 from 0.59 to
0.70. But McNemar's test could not distinguish any model from a TF-IDF+SVM baseline. Three flaws
explain that, and this notebook fixes all three.

| Flaw in run 1 | Consequence | Fix here |
|---|---|---|
| `MAX_LENGTH = 256` | Mean document is ~1,714 Sinhala characters; roughly two-thirds of every article was truncated before the model saw it | `MAX_LENGTH = 512` |
| Single 80/20 split | Every metric rested on 600 documents, and only 22 of them were genuinely FALSE/PARTIAL | **5-fold cross-validation**: every document gets an out-of-fold prediction, so metrics use all 3,000 and the strict subgroup uses all 110 |
| McNemar's test | Tests whether two models make the same *number* of errors. Our claim is about *where* the errors fall — a model can fix its class balance without changing its error count | **Paired bootstrap on macro-F1**, with McNemar retained and reported honestly as a secondary result |

### Structure

- **Stage A — screening.** All candidate models, one split, to pick finalists cheaply.
- **Stage B — confirmation.** Finalists + the corrupted-data control + a class-weighted variant,
  under 5-fold CV, with bootstrap confidence intervals.

### Before you run
- Runtime → Change runtime type → **T4 GPU**
- No manual uploads: both corpus files are downloaded from the official repository.
- Runtime → **Run all**. Budget roughly **90–120 minutes**. Stage A is ~35 min, Stage B ~60 min.

## 0 — Environment

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas openpyxl statsmodels matplotlib seaborn
print("Environment ready.")

In [ ]:
import os, re, json, random, warnings, time, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU.")

# ----------------------------------------------------------------- configuration
MAX_LENGTH    = 512      # was 256 - see the table above
EPOCHS        = 3
LEARNING_RATE = 2e-5
N_FOLDS       = 5
N_BOOTSTRAP   = 2000
TEST_SIZE     = 0.2      # Stage A screening split only

RESULTS_DIR = "results"; os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"max_len={MAX_LENGTH}  epochs={EPOCHS}  lr={LEARNING_RATE}  "
      f"folds={N_FOLDS}  bootstrap={N_BOOTSTRAP}  seed={SEED}")

# ---- torch.amp compatibility (the torch.cuda.amp spelling is deprecated)
def make_scaler():
    try:    return torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    except Exception: return torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def autocast():
    try:    return torch.amp.autocast("cuda", enabled=(device.type == "cuda"))
    except Exception: return torch.cuda.amp.autocast(enabled=(device.type == "cuda"))

## 1 — Data integrity check

The LIRNEasia repository distributes this corpus in **two formats**, with no warning attached to
either:

| File | Sinhala text | Notes |
|---|---|---|
| `Corpus.csv`  | destroyed — every codepoint replaced by `?` | the format most naturally consumed programmatically (`pd.read_csv`, published example code) |
| `Corpus.xlsx` | intact | same 3,000 documents, same order, same labels |

The hazard is that the failure is **silent**. A model trained from the CSV does not error; it
trains, reports plausible accuracy, and has learned nothing from the text. Section 6.1 quantifies
exactly what that costs by training the same architecture on both.

Both files are downloaded directly from the repository below, so this entire notebook is
reproducible by anyone with the URL.

In [ ]:
REPO     = "https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main"
CSV_URL  = f"{REPO}/Corpus.csv"     # the damaged distribution
XLSX_URL = f"{REPO}/Corpus.xlsx"    # the intact distribution

!wget -q -O Corpus_published.csv "$CSV_URL"
!wget -q -O Corpus.xlsx          "$XLSX_URL"

for f in ("Corpus_published.csv", "Corpus.xlsx"):
    assert os.path.exists(f) and os.path.getsize(f) > 0, f"download failed: {f}"
    print(f"{f:24s} {os.path.getsize(f)/1e6:6.2f} MB")
print("\nBoth distributions fetched from the official repository.")

In [ ]:
SINHALA_RE = re.compile(r"[඀-෿]")
df_pub   = pd.read_csv("Corpus_published.csv", encoding="latin-1")
df_clean = pd.read_excel("Corpus.xlsx")

aligned = (len(df_pub) == len(df_clean)
           and (df_pub["X1"].values == df_clean["X1"].values).all()
           and (df_pub["domain"].astype(str).values == df_clean["domain"].astype(str).values).all())
print("Row-for-row aligned:", aligned)
assert aligned, "Files are not aligned - the corrupted-vs-corrected control would be invalid."

sin = lambda s: len(SINHALA_RE.findall(str(s)))
qm  = lambda s: str(s).count("?")

integrity = pd.DataFrame({
    "file": ["Corpus.csv (published)", "Corpus.xlsx (intact)"],
    "documents": [len(df_pub), len(df_clean)],
    "mean Sinhala chars/doc": [df_pub["content"].apply(sin).mean(), df_clean["content"].apply(sin).mean()],
    "mean '?' chars/doc":     [df_pub["content"].apply(qm).mean(),  df_clean["content"].apply(qm).mean()],
}).round(1)

print("\n" + "="*74); print("DATA INTEGRITY COMPARISON"); print("="*74)
print(integrity.to_string(index=False))
print("\nSame document, both files:")
print("  published :", str(df_pub['content'].iloc[0])[:90])
print("  intact    :", str(df_clean['content'].iloc[0])[:90])
integrity.to_csv(f"{RESULTS_DIR}/data_integrity.csv", index=False)

## 2 — Labels and truncation audit

Excel coerced the string `"FALSE"` into a boolean, which silently drops those 27 documents from
any string comparison; `normalise_type` repairs that.

The second cell measures how much of each article actually reaches the model at 256 vs 512
tokens. This quantifies the truncation problem rather than assuming it.

In [ ]:
def normalise_type(v):
    if isinstance(v, bool): return "FALSE" if v is False else "TRUE"
    return str(v).strip().upper()

data = pd.DataFrame({
    "text_clean":     df_clean["content"].astype(str),
    "text_published": df_pub["content"].astype(str),
    "type":           df_clean["type"].apply(normalise_type),
    "domain":         df_clean["domain"].astype(str),
})
data = data[data["text_clean"].str.strip().str.len() > 0].reset_index(drop=True)
data["label"] = (data["type"] == "CREDIBLE").astype(int)

y_all       = data["label"].values
strict_all  = data["type"].isin(["FALSE", "PARTIAL"]).values
texts_clean = data["text_clean"].tolist()        # corrected text
texts_pub   = data["text_published"].tolist()    # corrupted text, same documents

print("Four-way labels:"); print(data["type"].value_counts().to_string())
print(f"\nBinary: {(y_all==1).sum()} CREDIBLE / {(y_all==0).sum()} NOT CREDIBLE   (n={len(data)})")
print(f"Genuine misinformation (FALSE/PARTIAL): {strict_all.sum()} documents")
print("\nUnder 5-fold CV every one of those gets an out-of-fold prediction,")
print("so the strict subgroup below uses all of them - not the ~22 a single split gave us.")

In [ ]:
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
_sample = data["text_clean"].sample(300, random_state=SEED).tolist()
_lens = np.array([len(_tok.encode(t, truncation=False)) for t in _sample])

print("Token length of documents (XLM-R tokenizer, 300-document sample)")
print(f"  median {np.median(_lens):.0f} | mean {_lens.mean():.0f} | 90th pct {np.percentile(_lens,90):.0f}")
print(f"\n  fully captured at 256 tokens : {(_lens<=256).mean()*100:5.1f}% of documents")
print(f"  fully captured at 512 tokens : {(_lens<=512).mean()*100:5.1f}% of documents")
print(f"  mean fraction of each document seen at 256 : {np.minimum(256/_lens,1).mean()*100:5.1f}%")
print(f"  mean fraction of each document seen at 512 : {np.minimum(512/_lens,1).mean()*100:5.1f}%")
del _tok; gc.collect()

## 3 — Evaluation helpers

`record()` stores metrics **and** per-document predictions, because the bootstrap and McNemar
tests both need per-document agreement, not just summary numbers.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

RESULTS, PREDICTIONS = {}, {}

def record(name, y_true, y_pred, note="", quiet=False):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")
    wf1 = f1_score(y_true, y_pred, average="weighted")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1],
                                                 average="binary", zero_division=0)
    RESULTS[name] = {"model": name, "note": note, "accuracy": acc, "macro_f1": mf1,
                     "weighted_f1": wf1, "credible_precision": p, "credible_recall": r,
                     "credible_f1": f}
    PREDICTIONS[name] = np.asarray(y_pred)
    if not quiet:
        print(f"\n{'='*74}\n{name}  {note}\n{'='*74}")
        print(f"Accuracy {acc:.4f} | Macro-F1 {mf1:.4f} | Weighted-F1 {wf1:.4f}")
        print(f"CREDIBLE  precision {p:.3f}  recall {r:.3f}  f1 {f:.3f}")
        print(classification_report(y_true, y_pred,
              target_names=["NOT CREDIBLE", "CREDIBLE"], zero_division=0))
    return RESULTS[name]

## 4 — Training harness

One function trains one model on one (train, test) index pair. Cross-validation calls it five
times; screening calls it once. Every finished model is moved off the GPU immediately — keeping
them resident is what caused the out-of-memory failures in the first run.

`class_weight=True` applies inverse-frequency weighting to the loss, which trades precision on
the majority class for recall on the minority one.

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=MAX_LENGTH, return_tensors="pt")
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def fit_predict(checkpoint, texts, labels, tr_idx, te_idx,
                batch_size=8, class_weight=False, keep_model=False, verbose=True):
    # Returns (predictions_for_te_idx, model_or_None). Returns (None, None) on failure.
    set_seed()
    try:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint, num_labels=2,
            id2label={0: "NOT_CREDIBLE", 1: "CREDIBLE"},
            label2id={"NOT_CREDIBLE": 0, "CREDIBLE": 1}).to(device)
    except Exception as e:
        print(f"  !! load failed for {checkpoint}: {type(e).__name__}: {e}")
        return None, None

    tr_texts = [texts[i] for i in tr_idx]; tr_y = labels[tr_idx]
    te_texts = [texts[i] for i in te_idx]; te_y = labels[te_idx]

    train_loader = DataLoader(TextDataset(tr_texts, tr_y, tokenizer), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TextDataset(te_texts, te_y, tokenizer), batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, total // 10, total)
    scaler = make_scaler()

    loss_fn = None
    if class_weight:
        n0, n1 = (tr_y == 0).sum(), (tr_y == 1).sum()
        w = torch.tensor([len(tr_y)/(2*n0), len(tr_y)/(2*n1)], dtype=torch.float).to(device)
        loss_fn = CrossEntropyLoss(weight=w)
        if verbose: print(f"  class weights: NOT_CREDIBLE {w[0]:.3f}  CREDIBLE {w[1]:.3f}")

    try:
        for epoch in range(EPOCHS):
            model.train(); running = 0.0
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                with autocast():
                    if loss_fn is None:
                        loss = model(**batch).loss
                    else:
                        logits = model(input_ids=batch["input_ids"],
                                       attention_mask=batch["attention_mask"]).logits
                        loss = loss_fn(logits, batch["labels"])
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
                running += loss.item()
            if verbose: print(f"    epoch {epoch+1}/{EPOCHS}  mean loss {running/len(train_loader):.4f}")
    except torch.cuda.OutOfMemoryError:
        print(f"  !! OOM at batch_size={batch_size}")
        del model, optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
        return None, None

    model.eval(); preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            preds.extend(logits.argmax(-1).cpu().numpy())

    out_model = None
    if keep_model:
        out_model = (model.to("cpu"), tokenizer)
    del model, optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
    return np.asarray(preds), out_model

print("Harness ready.")

## 5 — Stage A: screening

One stratified 80/20 split, every candidate, at 512 tokens. The point is to rank models cheaply
so that the expensive cross-validation only runs on the ones worth confirming.

Candidates follow *BERTifying Sinhala* (Dhananjaya et al., LREC 2022), which benchmarked
pretrained models for Sinhala text classification.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

idx_tr, idx_te = train_test_split(np.arange(len(data)), test_size=TEST_SIZE,
                                  random_state=SEED, stratify=y_all)
y_screen = y_all[idx_te]

CANDIDATES = [
    ("xlm-roberta-base",             "XLM-R base",     8),
    ("bert-base-multilingual-cased", "mBERT",          8),
    ("setu4993/LaBSE",               "LaBSE",          8),
    ("NLPC-UOM/SinBERT-small",       "SinBERT-small",  8),
    ("NLPC-UOM/SinBERT-large",       "SinBERT-large",  4),
    ("xlm-roberta-large",            "XLM-R large",    2),
]

screen = []
for ckpt, label, bs in CANDIDATES:
    print(f"\n{'#'*70}\n# SCREENING: {label}  ({ckpt})\n{'#'*70}")
    t0 = time.time()
    preds, _ = fit_predict(ckpt, texts_clean, y_all, idx_tr, idx_te, batch_size=bs)
    if preds is None:
        print(f"  {label}: FAILED"); screen.append({"model": label, "checkpoint": ckpt,
              "macro_f1": np.nan, "accuracy": np.nan, "minutes": np.nan}); continue
    mf1 = f1_score(y_screen, preds, average="macro")
    acc = accuracy_score(y_screen, preds)
    mins = (time.time()-t0)/60
    print(f"  -> {label}: macro-F1 {mf1:.4f} | accuracy {acc:.4f} | {mins:.1f} min")
    screen.append({"model": label, "checkpoint": ckpt, "macro_f1": mf1,
                   "accuracy": acc, "minutes": round(mins,1)})

screen_df = pd.DataFrame(screen).sort_values("macro_f1", ascending=False, na_position="last")
print("\n" + "="*74); print("STAGE A - SCREENING RESULTS (single split, 512 tokens)"); print("="*74)
print(screen_df.to_string(index=False))
screen_df.to_csv(f"{RESULTS_DIR}/stage_a_screening.csv", index=False)

FINALISTS = screen_df.dropna(subset=["macro_f1"]).head(2)[["checkpoint","model"]].values.tolist()
print(f"\nFinalists for cross-validation: {[f[1] for f in FINALISTS]}")

## 6 — Stage B: 5-fold cross-validation

Every document receives one out-of-fold prediction, so the pooled predictions cover all 3,000
documents — including all 110 FALSE/PARTIAL ones. Four configurations are confirmed:

1. **Each finalist** on corrected text.
2. **The top finalist with class weighting**, to see whether the minority-class recall problem is
   fixable at the loss function rather than by changing model.
3. **The corrupted-data control** — same architecture, same folds, published text. This is the
   comparison that isolates data quality.

In [ ]:
# Fold definition lives in its own cell so that the classical baselines (section 7)
# can be cross-validated over the identical folds even if a transformer run fails.
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(np.zeros(len(y_all)), y_all))

for k, (tr, te) in enumerate(FOLDS, 1):
    print(f"fold {k}: train {len(tr)}  test {len(te)}  "
          f"(test CREDIBLE {int(y_all[te].sum())}, FALSE/PARTIAL {int(strict_all[te].sum())})")
print(f"\nTotal out-of-fold predictions per configuration: {sum(len(te) for _, te in FOLDS)}")

In [ ]:
def cross_validate(checkpoint, label, texts, batch_size=8, class_weight=False, note=""):
    pooled = np.full(len(y_all), -1, dtype=int)
    per_fold = []
    print(f"\n{'#'*70}\n# CV: {label}\n{'#'*70}")
    for k, (tr, te) in enumerate(FOLDS, 1):
        print(f"  fold {k}/{N_FOLDS}")
        preds, _ = fit_predict(checkpoint, texts, y_all, tr, te,
                               batch_size=batch_size, class_weight=class_weight, verbose=False)
        if preds is None:
            print(f"  !! fold {k} failed - aborting {label}"); return None
        pooled[te] = preds
        f = f1_score(y_all[te], preds, average="macro"); per_fold.append(f)
        print(f"    fold macro-F1 {f:.4f}")
    print(f"  -> {label}: macro-F1 {np.mean(per_fold):.4f} +/- {np.std(per_fold):.4f} across folds")
    record(label, y_all, pooled, note=note, quiet=True)
    RESULTS[label]["fold_mean"] = float(np.mean(per_fold))
    RESULTS[label]["fold_std"]  = float(np.std(per_fold))
    RESULTS[label]["checkpoint"] = checkpoint
    return pooled

CV_JOBS = []
for ckpt, label in FINALISTS:
    CV_JOBS.append((ckpt, f"{label} [CV]", texts_clean, False, "(corrected data, 5-fold CV)"))
top_ckpt, top_label = FINALISTS[0]
CV_JOBS.append((top_ckpt, f"{top_label} + class weights [CV]", texts_clean, True,
                "(corrected data, class-weighted loss)"))
CV_JOBS.append(("xlm-roberta-base", "XLM-R base [CORRUPTED, CV]", texts_pub, False,
                "(control: published Corpus.csv - Sinhala destroyed)"))

for ckpt, label, txt, cw, note in CV_JOBS:
    cross_validate(ckpt, label, txt, batch_size=4 if "large" in ckpt else 8,
                   class_weight=cw, note=note)

## 7 — Classical baselines, same folds

The baselines are cross-validated over the identical folds, so every number in the results table
comes from the same 3,000 out-of-fold predictions and the comparisons are paired.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

MAJORITY = int(np.bincount(y_all).argmax())
record("Majority class", y_all, np.full(len(y_all), MAJORITY),
       note="(predicts the most frequent label for every document)", quiet=True)

for name, make in [("TF-IDF + Naive Bayes", lambda: MultinomialNB()),
                   ("TF-IDF + Linear SVM",  lambda: LinearSVC(class_weight="balanced", random_state=SEED))]:
    pooled = np.full(len(y_all), -1, dtype=int)
    for tr, te in FOLDS:
        vec = TfidfVectorizer(max_features=5000, sublinear_tf=True)
        Xtr = vec.fit_transform([texts_clean[i] for i in tr])
        Xte = vec.transform([texts_clean[i] for i in te])
        pooled[te] = make().fit(Xtr, y_all[tr]).predict(Xte)
    record(name, y_all, pooled, note="(classical baseline, 5-fold CV)", quiet=True)
    print(f"{name}: macro-F1 {RESULTS[name]['macro_f1']:.4f}")

## 8 — Results

In [ ]:
cols = ["model","note","accuracy","macro_f1","weighted_f1",
        "credible_precision","credible_recall","credible_f1"]
res = pd.DataFrame(RESULTS).T
for c in cols[2:]: res[c] = res[c].astype(float).round(4)
if "fold_std" in res.columns:
    res["fold_std"] = res["fold_std"].astype(float).round(4)
    cols = cols + ["fold_std"]
res = res[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True)

print("="*110)
print(f"RESULTS  -  pooled out-of-fold predictions, n={len(y_all)}, {N_FOLDS}-fold CV, {MAX_LENGTH} tokens")
print("="*110)
print(res.to_string(index=False))
res.to_csv(f"{RESULTS_DIR}/results.csv", index=False)

best_model = res.iloc[0]["model"]
print(f"\n>>> Best by macro-F1: {best_model}")

## 9 — Statistical validation

**Primary test — paired bootstrap on macro-F1.** Resample the 3,000 documents with replacement
2,000 times; each resample recomputes both models' macro-F1 on the *same* resampled documents and
takes the difference. The resulting distribution gives a 95% confidence interval and a one-sided
p-value. This tests the quantity we actually claim.

**Secondary test — McNemar.** Reported for completeness. It asks whether two models make the same
*number* of errors, which is a different question: a model can correct its class balance without
changing its error count, and that is exactly what happened in run 1.

In [ ]:
rng = np.random.default_rng(SEED)
BOOT_IDX = rng.integers(0, len(y_all), size=(N_BOOTSTRAP, len(y_all)))

def bootstrap_gap(a, b):
    pa, pb = PREDICTIONS[a], PREDICTIONS[b]
    diffs = np.empty(N_BOOTSTRAP)
    for i in range(N_BOOTSTRAP):
        s = BOOT_IDX[i]
        diffs[i] = (f1_score(y_all[s], pa[s], average="macro")
                    - f1_score(y_all[s], pb[s], average="macro"))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = float((diffs <= 0).mean())          # one-sided: P(a is not better than b)
    return {"model_A": a, "model_B": b,
            "macroF1_gap": round(float(np.mean(diffs)), 4),
            "CI95_low": round(float(lo), 4), "CI95_high": round(float(hi), 4),
            "p_value": p, "significant": bool(lo > 0)}

others = [m for m in PREDICTIONS if m != best_model]
print(f"Bootstrapping {len(others)} comparisons x {N_BOOTSTRAP} resamples - this takes a minute...")
boot = pd.DataFrame([bootstrap_gap(best_model, m) for m in others]) \
         .sort_values("macroF1_gap", ascending=False)
boot["p_value"] = boot["p_value"].apply(lambda p: f"{p:.4f}" if p >= 1e-4 else "<0.0001")

print("\n" + "="*110)
print(f"PAIRED BOOTSTRAP ON MACRO-F1  -  '{best_model}' vs each alternative")
print("="*110)
print(boot.to_string(index=False))
boot.to_csv(f"{RESULTS_DIR}/bootstrap_tests.csv", index=False)
print("\nsignificant = the 95% CI for the macro-F1 gap excludes zero.")

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(a, b):
    ca = (PREDICTIONS[a] == y_all); cb = (PREDICTIONS[b] == y_all)
    tbl = np.array([[np.sum(ca & cb), np.sum(ca & ~cb)],
                    [np.sum(~ca & cb), np.sum(~ca & ~cb)]])
    r = mcnemar(tbl, exact=((tbl[0,1]+tbl[1,0]) < 25), correction=True)
    return {"model_A": a, "model_B": b,
            "A_right_B_wrong": int(tbl[0,1]), "B_right_A_wrong": int(tbl[1,0]),
            "p_value": float(r.pvalue), "significant": bool(r.pvalue < 0.05)}

sig = pd.DataFrame([mcnemar_test(best_model, m) for m in others])
sig["p_value"] = sig["p_value"].apply(lambda p: f"{p:.2e}" if p < 1e-3 else f"{p:.4f}")
print("="*110); print(f"McNEMAR (secondary)  -  '{best_model}' vs each alternative"); print("="*110)
print(sig.to_string(index=False))
sig.to_csv(f"{RESULTS_DIR}/mcnemar_tests.csv", index=False)

## 10 — Strict fake-news subgroup

With pooled cross-validation these rates are computed over **all 110** FALSE/PARTIAL documents
rather than the ~22 that landed in a single test split.

Read `detection rate` together with `CREDIBLE recall`: a model that labels everything NOT
CREDIBLE scores 1.0 detection and 0.0 recall, and is useless. `balanced` is the mean of the two.

In [ ]:
n_strict = int(strict_all.sum())
cred = (y_all == 1)
rows = []
for name, preds in PREDICTIONS.items():
    caught = int((preds[strict_all] == 0).sum())
    kept   = float((preds[cred] == 1).mean())
    rows.append({"model": name, "FALSE/PARTIAL docs": n_strict, "correctly flagged": caught,
                 "detection rate": round(caught/n_strict, 4), "CREDIBLE recall": round(kept, 4),
                 "balanced": round((caught/n_strict + kept)/2, 4)})
strict = pd.DataFrame(rows).sort_values("balanced", ascending=False)
print("="*100); print(f"DETECTION ON GENUINE MISINFORMATION (n={n_strict}, pooled out-of-fold)"); print("="*100)
print(strict.to_string(index=False))
strict.to_csv(f"{RESULTS_DIR}/strict_subgroup.csv", index=False)

## 11 — Figures

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns

plot_df = res.sort_values("macro_f1")
fig, ax = plt.subplots(figsize=(10, 0.55*len(plot_df)+2))
colors = ["#c0392b" if "CORRUPTED" in m else "#7f8c8d" if ("Majority" in m or "TF-IDF" in m)
          else "#2e86c1" for m in plot_df["model"]]
ax.barh(plot_df["model"], plot_df["macro_f1"].astype(float), color=colors)
ax.axvline(float(RESULTS["Majority class"]["macro_f1"]), ls="--", c="k", lw=1, label="majority-class floor")
ax.set_xlabel("Macro F1 (pooled 5-fold CV)"); ax.set_title("Module 1 — comparative model performance")
ax.legend(); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/model_comparison.png", dpi=160); plt.show()

show = [m for m in PREDICTIONS if "CORRUPTED" in m or m == best_model][:2]
fig, axes = plt.subplots(1, len(show), figsize=(5.5*len(show), 4.5))
if len(show) == 1: axes = [axes]
for ax, name in zip(axes, show):
    cm = confusion_matrix(y_all, PREDICTIONS[name])
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["NOT CRED","CREDIBLE"], yticklabels=["NOT CRED","CREDIBLE"])
    ax.set_title(f"{name}\nmacroF1={RESULTS[name]['macro_f1']:.3f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/confusion_matrices.png", dpi=160); plt.show()

## 12 — Summary to paste back into the chat

In [ ]:
print("="*110); print("SINHALACHECK MODULE 1 - RUN SUMMARY (v2: 512 tokens, 5-fold CV, bootstrap)")
print("="*110)
print(f"\nn={len(y_all)} documents, {N_FOLDS}-fold CV, max_length={MAX_LENGTH}")
print(f"Majority-class floor: macro-F1 {RESULTS['Majority class']['macro_f1']:.4f}\n")
print(res.to_string(index=False))
print("\n" + "-"*110); print("STAGE A SCREENING"); print("-"*110)
print(screen_df.to_string(index=False))
print("\n" + "-"*110); print("PAIRED BOOTSTRAP ON MACRO-F1 (primary test)"); print("-"*110)
print(boot.to_string(index=False))
print("\n" + "-"*110); print("McNEMAR (secondary test)"); print("-"*110)
print(sig.to_string(index=False))
print("\n" + "-"*110); print("STRICT FAKE-NEWS SUBGROUP"); print("-"*110)
print(strict.to_string(index=False))
print("\n" + "-"*110); print("DATA INTEGRITY"); print("-"*110)
print(integrity.to_string(index=False))
print("\n" + "="*110); print("Copy everything above and paste it back into the chat."); print("="*110)

## 13 — Fit and save the final model

Cross-validation measures a *configuration*; it does not leave behind a single deployable model.
So the winning configuration is refit once on the Stage A training split and saved in the layout
`app.py` already expects — with `id2label` written into the config this time.

Every step here is non-fatal: a Drive mount failure must never destroy a completed run.

In [ ]:
FINAL_CKPT = RESULTS[best_model].get("checkpoint", FINALISTS[0][0])
FINAL_CW   = "class weights" in best_model
print(f"Refitting '{best_model}'  (checkpoint={FINAL_CKPT}, class_weight={FINAL_CW})")

_, kept = fit_predict(FINAL_CKPT, texts_clean, y_all, idx_tr, idx_te,
                      batch_size=4 if "large" in FINAL_CKPT else 8,
                      class_weight=FINAL_CW, keep_model=True)

OUT, drive_ok = "/content/SinhalaCheck_model_v3", False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/SinhalaCheck_model_v3"; drive_ok = True
    print("Drive mounted.")
except Exception as e:
    print(f"!! Drive mount failed ({type(e).__name__}). Saving locally to {OUT}.")
    print("   Retrieve it from the folder icon in the left sidebar.")

os.makedirs(OUT, exist_ok=True)
if kept is not None:
    m, tk = kept; m.save_pretrained(OUT); tk.save_pretrained(OUT)
    print(f"Saved -> {OUT}")
else:
    print("!! Refit failed; no model saved.")

manifest = {"selected_model": best_model, "checkpoint": FINAL_CKPT, "class_weighted": FINAL_CW,
            "seed": SEED, "max_length": MAX_LENGTH, "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE, "n_folds": N_FOLDS, "n_documents": int(len(y_all)),
            "label_map": {"0": "NOT_CREDIBLE", "1": "CREDIBLE"},
            "results": {k: {kk: (float(vv) if isinstance(vv,(int,float,np.floating)) else vv)
                            for kk, vv in v.items()} for k, v in RESULTS.items()}}
with open(f"{RESULTS_DIR}/run_manifest.json","w") as f: json.dump(manifest, f, indent=2, ensure_ascii=False)

if drive_ok:
    !cp -r $RESULTS_DIR /content/drive/MyDrive/SinhalaCheck_results
    print("Results copied to Drive.")
else:
    !zip -qr sinhalacheck_results.zip $RESULTS_DIR
    print("Results zipped -> sinhalacheck_results.zip")
    try:
        from google.colab import files as _f; _f.download("sinhalacheck_results.zip")
    except Exception as e:
        print(f"   (auto-download unavailable: {type(e).__name__}) - use the Files sidebar.")

## 14 — Optional: publish the model to Hugging Face Hub

Colab deletes `/content` when the runtime ends, which is how the previous copy of this model was
lost. Pushing to the Hub gives it a permanent address, and solves a second problem too: the model
is ~1.9GB and GitHub rejects files over 100MB, so the project repository can reference the Hub
instead of trying to store weights.

**Setup (once):** create a free account at huggingface.co, then
Settings -> Access Tokens -> Create new token -> type **Write**.

Set `HF_USERNAME` below and run. Leave it blank to skip.

In [ ]:
HF_USERNAME = ""          # <-- your huggingface.co username, e.g. "kaweeshwara"
REPO_NAME   = "sinhalacheck-module1"

if not HF_USERNAME:
    print("HF_USERNAME is blank - skipping the Hub upload.")
    print("The model is still saved at:", OUT)
elif kept is None:
    print("No model was refit in section 13, so there is nothing to push.")
else:
    !pip install -q huggingface_hub
    from huggingface_hub import login, create_repo
    login()                                        # paste your WRITE token at the prompt
    repo_id = f"{HF_USERNAME}/{REPO_NAME}"
    create_repo(repo_id, exist_ok=True, private=False)

    m, tk = kept
    m.push_to_hub(repo_id); tk.push_to_hub(repo_id)

    print(f"\nPublished -> https://huggingface.co/{repo_id}")
    print("\nAnything that needs the model can now load it directly:")
    print(f'  AutoModelForSequenceClassification.from_pretrained("{repo_id}")')
    print("\nPut that line in the README instead of committing the weights.")